# Object Detection
Using the MMdetection3D package, we have a wide range of object detection networks we can use.
For our case on incomplete partial rgb scans, we use votenet

## Loading the pointcloud
The model expects ??

In [12]:
# the path of the pointcloud
dataPath = "../../mmdetection3d/demo/data/sunrgbd/000017.bin"
#dataPath = "/home/jvermandere/projects/mmdetection3d/demo/data/scannet/scene0000_00.bin"
demoFile = "../../mmdetection3d/demo/pcd_demo.py"
configFile = "../../mmdetection3d/configs/votenet/votenet_8xb16_sunrgbd-3d.py"
weightsFile = "/home/jvermandere/projects/DRM/checkpoints/votenet_16x8_sunrgbd-3d-10class_20210820_162823-bf11f014.pth"

!python {demoFile} {dataPath} {configFile} {weightsFile}

/home/jvermandere/.conda/envs/openmmlab/lib/python3.8/site-packages/mmcv/cnn/bricks/conv_module.py:208: UserWarning: Unnecessary conv bias before batch/instance norm
  warnings.warn(
Loads checkpoint by local backend from path: /home/jvermandere/projects/DRM/checkpoints/votenet_16x8_sunrgbd-3d-10class_20210820_162823-bf11f014.pth
02/25 16:41:29 - mmengine - WARNING - Failed to search registry with scope "mmdet3d" in the "function" registry tree. As a workaround, the current "function" registry in "mmengine" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmdet3d" is a correct scope, or whether the registry is initialized.
/home/jvermandere/.conda/envs/openmmlab/lib/python3.8/site-packages/mmengine/visualization/visualizer.py:196: UserWarning: Failed to add <class 'mmengine.visualization.vis_backend.LocalVisBackend'>, please provide the `save_dir` argument.
  warnings.warn(f'Failed to add {vis_backend.__class__}, '
/home

In [ ]:
import trimesh
import numpy as np
import json

jsonPath = "/home/jvermandere/projects/DRM/notebooks/outputs/preds/000017.json"
#jsonPath = "/home/jvermandere/projects/DRM/notebooks/outputs/preds/scene0000_00.json"

with open(jsonPath) as f:
    data = json.load(f)

# Parameters
score_threshold = 0.05  # Only show boxes with score > threshold

# Colormap for labels
label_colors = [
    [1, 0, 0, 0.5],  # red, alpha 0.5
    [0, 1, 0, 0.5],  # green
    [0, 0, 1, 0.5],  # blue
    [1, 1, 0, 0.5],  # yellow
    [1, 0, 1, 0.5],  # magenta
    [0, 1, 1, 0.5],  # cyan
]

def create_trimesh_box(bottom_center, size, rotation_z=0.0, color=[1,0,0,0.5]):
    """
    Create a trimesh Box mesh with given center, size, rotation, and color.
    """
    # Box is created centered at origin
    box = trimesh.creation.box(extents=size, transform=None)
    
    # Rotation matrix around z-axis
    c, s = np.cos(rotation_z), np.sin(rotation_z)
    R = np.array([
        [c, -s, 0, 0],
        [s,  c, 0, 0],
        [0,  0, 1, 0],
        [0,  0, 0, 1]
    ])
    
    # Translation to center
    T = np.eye(4)
    T[:3, 3] = [
    bottom_center[0],
    bottom_center[1],
    bottom_center[2] + size[2] / 2.0
    ]

    # Apply transform
    box.apply_transform(T @ R)
    
    # Set color (RGBA)
    box.visual.face_colors = color
    
    return box

def load_bin_pointcloud(file_path):
    """Load KITTI-style .bin point cloud"""
    points = np.fromfile(file_path, dtype=np.float32).reshape(-1, 6)  # x, y, z, rgb
    cloud = trimesh.points.PointCloud(points[:, :3], colors=points[:, 3:6]/255)
    return cloud
# Create list of meshes
meshes = []

for label, score, box in zip(data["labels_3d"], data["scores_3d"], data["bboxes_3d"]):
    if score < score_threshold:
        continue

    center = np.array(box[:3])
    size = np.array(box[3:6])
    rotation_z = box[6]
    color = label_colors[label % len(label_colors)]
    
    mesh = create_trimesh_box(center, size, rotation_z, color)
    meshes.append(mesh)

pcd = load_bin_pointcloud(dataPath)
meshes.append(pcd)

# Combine meshes for visualization
scene = trimesh.Scene(meshes)

# Show interactive visualization
scene.show()

{'labels_3d': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], 'scores_3d': [0.00013980150106362998, 3.111652040388435e-05, 0.0002225449134130031, 0.9950772523880005, 4.837020242121071e-07, 0.0003394779050722718, 0.00481010926887393, 2.1470788851729594e-05, 0.0016859677853062749, 0.005460729356855154, 0.011171950958669186, 0.00808151625096798, 0.0013939813943579793, 1.1477354746602941e-06, 0.0001106403797166422, 0.005675558932125568, 0.003756413236260414, 0.004461848642677069, 0.00047185461153276265, 0.020177999511361122, 6.432899681385607e-05, 0.0002281550841871649, 0.00010237879905616865, 4.1310046071885154e-05, 1.9094914094353044e-08, 1.4380273569258861e-05, 0.00019287411123514175, 1.3212639714765828e-05, 2.8956175810890272e-05, 0.265